# Nigerian Equipment Marketplace Analysis

## Project Overview

This project has three parts, built on one dataset: **Task 1** web scraping and data collection, **Task 2** exploratory data analysis, and **Task 3** data visualization. This notebook covers Task 1.

**Business Question (whole project):** What does the marketplace's listed equipment inventory reveal about category representation, geographic concentration, and advertised price availability, and how can these insights support marketplace planning?

## Task 1: Web Scraping and Data Collection

**Objective:** Collect publicly available listings from the Nigerian Equipment Marketplace, check what the data contains, and save a clean raw dataset for the analysis in Task 2.

**Approach:**
- Inspect how the site loads its listings and identify the public listings API behind it.
- Collect every page of approved listings (245 listings across 25 pages).
- Save only the fields needed for the analysis. Seller contact and owner details are left out for privacy.

**Output:** `data/equipment_listings_raw.json`, which is the input for Task 2.


### Inspecting the Website:
Look at the page’s HTML with requests and BeautifulSoup. Searching for a heading returns None, so the listings are loaded by JavaScript and are not in the static HTML.

In [1]:
import requests

In [2]:
url = "https://equipmentmarket.ng/"
response = requests.get(url, timeout=20)

print(response.status_code)

200


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>Equipment Marketplace</title>


In [4]:
print(soup.find("h3"))

None


In [5]:
print(response.text[:2000])

<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <!-- TODO: Set the document title to the name of your application -->
    <title>Equipment Marketplace</title>
    <meta name="description" content="Nigeria&#39;s trusted marketplace for verified equipment. Connect with OEMs, lessors, lessees, and service providers â all in one place.">
    <meta name="author" content="Equipment Marketplace" />

    
    
    <meta property="og:type" content="website" />
    

    <meta name="twitter:card" content="summary_large_image" />
    <meta name="twitter:site" content="@Lovable" />
    
    <meta property="og:title" content="Equipment Marketplace">
  <meta name="twitter:title" content="Equipment Marketplace">
  <meta property="og:description" content="Nigeria&#39;s trusted marketplace for verified equipment. Connect with OEMs, lessors, lessees, and service providers â all in one place.">
  <

### Checking robots.txt
The site’s robots.txt allows crawling of all paths for all user agents.

In [6]:
robots_url = "https://equipmentmarket.ng/robots.txt"
robots_response = requests.get(robots_url, timeout=20)

print(robots_response.status_code)
print(robots_response.text[:3000])

200
User-agent: Googlebot
Allow: /

User-agent: Bingbot
Allow: /

User-agent: Twitterbot
Allow: /

User-agent: facebookexternalhit
Allow: /

User-agent: *
Allow: /



### Finding the Listings API: 
Search the site’s JavaScript files for the endpoint behind the listings. The next cells print parts of that code to trace it.

In [7]:
js_url = "https://equipmentmarket.ng/assets/index-NxSyl00x.js"
js_response = requests.get(js_url, timeout=20)

print(js_response.status_code)
print(js_response.text[:1000])

200
const __vite__mapDeps=(i,m=__vite__mapDeps,d=(m.f||(m.f=["assets/Index-CL5gKvSt.js","assets/Footer-BpKIIosi.js","assets/UserAvatar-aQK8mkJ1.js","assets/contentApi-DJMdRSGk.js","assets/useAuthSession-Bl4ck8ok.js","assets/useMyUserDataQuery-NVtPnK_h.js","assets/queryKeys-ClvnXOu1.js","assets/useWorkflowQuery-DrjI5zWS.js","assets/sessionData-D3Uur9gn.js","assets/AdSlot-CJdxasw0.js","assets/adApi-Cuoeh0fY.js","assets/map-pin-CcQxE2yS.js","assets/search-BvrQx8im.js","assets/EquipmentCard-Yg5eoc0R.js","assets/badge-DuegYin5.js","assets/share-2-C6E9VxtY.js","assets/copy-DmSiCHVF.js","assets/mail-CzwMrx32.js","assets/priceDisplay-D0jrbmhT.js","assets/shield-CboihhQk.js","assets/eye-DaPlSaCP.js","assets/liveMarketplace-DCm9mIdN.js","assets/equipmentSpecifications-ff83feaI.js","assets/listingCategories-mOVKeq5L.js","assets/chevron-right-C81t9NDr.js","assets/card-Yt189hRV.js","assets/truck-BtQL6130.js","assets/star-DoB0k6wI.js","assets/phone-0Xx3BEpF.js","assets/handshake-DaERzAkR.js","assets

In [8]:
keywords = ["supabase", "api/", "equipment-listings", "fetch(", "liveMarketplace"]

for keyword in keywords:
    print(keyword, js_response.text.find(keyword))

supabase -1
api/ 330623
equipment-listings -1
fetch( 7733
liveMarketplace 704


In [9]:
position = js_response.text.find("api/")

print(js_response.text[position - 300:position + 500])

ion")||"anonymous"}`},Bn=async(e,t)=>{const n=String((t==null?void 0:t.method)||"GET").toUpperCase(),r=g2(e,t),s=n==="GET";if(s){const o=Kl.get(r);if(o&&o.expiresAt>Date.now())return o.promise}else Kl.clear();const i=(async()=>{const o=yn(Vv);let a=null;for(const l of o)try{const c=await fetch(Ae(`/api/auth${e}`,l),t);if(c.status===404&&l!==o[o.length-1]){a=new Error("Authentication route not found.");continue}const u=await m2(c);return Vv=l,u}catch(c){a=c instanceof Error?c:new Error("Authentication request failed.")}throw a||new Error("Authentication request failed.")})();return s&&(Kl.set(r,{expiresAt:Date.now()+h2,promise:i}),i.catch(()=>{Kl.delete(r)})),i},qz=async(e,t)=>Bn("/login",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({email:e,password:t})}),


In [10]:
module_url = "https://equipmentmarket.ng/assets/liveMarketplace-DCm9mIdN.js"
module_response = requests.get(module_url, timeout=20)

print(module_response.status_code)
print(module_response.text[:2000])

200
import{ar as P,i as w,as as $,c as I,P as R,at as W,g as V}from"./index-NxSyl00x.js";import{p as D}from"./equipmentSpecifications-ff83feaI.js";import{g as G,m as j,c as J}from"./listingCategories-mOVKeq5L.js";const v=t=>["true","1","yes"].includes(String(t||"").trim().toLowerCase()),k=5*60*60*1e3,E=35,K=50,M="elra:equipment:view-boost-v1",Y=()=>Math.floor(Math.random()*(K-E+1))+E,p=(t,i="")=>{if(!t)return t;const e=t.match(/\/(day|week|month|6 months|year)/i),n=e?e[0].toLowerCase():i,o=t.replace(/[₦N,\s]/gi,"").replace(/\/(day|week|month|6 months|year).*/i,""),r=Number(o);return!Number.isFinite(r)||r===0?t:`₦${Math.round(r).toLocaleString()}${n}`},a=(t,i=0)=>{const e=Number(String(t||"").replace(/[^0-9.-]/g,""));return Number.isFinite(e)?e:i},z=()=>{if(typeof window>"u")return{lastAppliedAt:Date.now(),boostsByListing:{}};try{const t=window.localStorage.getItem(M);if(!t)return{lastAppliedAt:Date.now(),boostsByListing:{}};const i=JSON.parse(t),e=i.boostsByListing&&typeof i.boostsByLi

In [11]:
module_text = module_response.text

for keyword in ["fetch(", "/api/", "equipment", "listings"]:
    print(keyword, module_text.find(keyword))

fetch( 9572
/api/ 9581
equipment 106
listings 3090


In [12]:
position = module_text.find("/api/")

print(module_text[position - 500:position + 1000])

|e.title||"General service").trim()||"General service",location:String(e.location||"Nigeria").trim()||"Nigeria",rating:a(e.rating,5),reviews:a(e.reviews,0),views:a(e.views,0),avatar:o.charAt(0).toUpperCase(),image:N(e),verified:String(e.verificationStatus||"").trim()==="Verified"||!!String(e.elraRegistrationId||"").trim(),featured:v(e.isFeatured),userKey:n,contactName:s.name,contactPhone:s.phone,contactWhatsapp:String(e.contactWhatsapp||e.whatsappNumber||"").trim()}});try{const e=await fetch($("/api/platform/logistics/public-drivers"));if(e.ok){const n=await e.json();n.success&&Array.isArray(n.drivers)&&i.push(...n.drivers)}}catch(e){console.warn("Failed to fetch public logistics drivers:",e)}return(t==null?void 0:t.fallbackToDemo)===!1,i},oe=()=>{const t=_();S();const i=G();return(i.length>0?i:I.map(n=>n.name)).map((n,o)=>{const r=I.find(s=>s.name===n);return{id:(r==null?void 0:r.id)||`dynamic-category-${o+1}`,name:n,icon:(r==null?void 0:r.icon)||"📦",count:t.filter(s=>j({category:Stri

In [13]:
import re

api_paths = re.findall(r'["\'`](/api/[^"\'` ]*)', module_text)

for path in sorted(set(api_paths)):
    print(path)

/api/platform/logistics/public-drivers


In [14]:
card_url = "https://equipmentmarket.ng/assets/EquipmentCard-Yg5eoc0R.js"
card_response = requests.get(card_url, timeout=20)

print(card_response.status_code)

200


In [15]:
card_text = card_response.text

for keyword in ["fetch(", "/api/", "equipment-listings", "views"]:
    print(keyword, card_text.find(keyword))

fetch( -1
/api/ -1
equipment-listings -1
views 3737


In [16]:
page_url = "https://equipmentmarket.ng/assets/Index-CL5gKvSt.js"
page_response = requests.get(page_url, timeout=20)

print(page_response.status_code)

200


In [17]:
page_text = page_response.text

for keyword in ["liveMarketplace", "fetch(", "/api/", "equipment"]:
    print(keyword, page_text.find(keyword))

liveMarketplace 395
fetch( -1
/api/ -1
equipment 1272


In [18]:
position = page_text.find("liveMarketplace")
print(page_text[position - 500:position + 800])

In [19]:
print("Length:", len(page_text))
print("Position:", page_text.find("liveMarketplace"))
print("Start:", page_text[:500])

Length: 23822
Position: 395
Start: import{u as M,r as n,j as e,c as k,B as w,L as p,s as T}from"./index-NxSyl00x.js";import{C as z,N as $,F as H}from"./Footer-BpKIIosi.js";import{M as q}from"./map-pin-CcQxE2yS.js";import{S as A}from"./search-BvrQx8im.js";import{E}from"./EquipmentCard-Yg5eoc0R.js";import{A as v}from"./AdSlot-CJdxasw0.js";import{getApprovedEquipmentListingsAsync as P,getApprovedServiceProvidersAsync as I}from"./liveMarketplace-DCm9mIdN.js";import{m as F}from"./listingCategories-mOVKeq5L.js";import{C as B}from"./che


In [20]:
position = module_text.find("getApprovedEquipmentListingsAsync")
print("Position:", position)
print(module_text[position - 300:position + 1800])

Position: 11779
ategory:String(n.category||"Service").trim()||"Service",location:String(n.location||"Nigeria").trim()||"Nigeria",price:String(n.pricePerDay||n.price||n.purchasePrice||"Contact provider").trim()||"Contact provider",imageUrl:N(n)})}),S(),[...t.values()]};export{_ as getApprovedEquipmentListings,ne as getApprovedEquipmentListingsAsync,Z as getApprovedServiceProviders,re as getApprovedServiceProvidersAsync,oe as getLiveEquipmentCategories,se as getLiveServiceCategoryMeta,ce as getLiveStakeholderProfiles,f as getStakeholderSlug};
//# sourceMappingURL=liveMarketplace-DCm9mIdN.js.map



In [21]:
print(module_text[7000:10000])

uipmentSpecifications||e.specifications||"")),isBnpl:v(e.isBnpl),isFeatured:v(e.isFeatured),views:a(e.views,0),equipmentId:String(e.equipmentId||r).trim()||r,registrationStatus:String(e.elraRegistrationId||"").trim()?"Registered":"Not registered",verificationStatus:String(e.verificationStatus||"").trim()==="Verified"?"Verified":String(e.elraRegistrationId||"").trim()?"Pending Review":"Not verified",ownerUserKey:n,ownerName:h(n),contactName:s.name,contactPhone:s.phone,provideOperator:String(e.provideOperator||"").toLowerCase()==="yes",operatorPolicy:String(e.operatorPolicy||"")==="external-allowed"?"external-allowed":"owner-only",escrowAvailable:O,leaseOptions:C,interactionStats:F(e)}});return(t==null?void 0:t.fallbackToDemo)===!1,B(i)},Z=()=>{const t=g.filter(i=>String(i.listingType||"equipment")==="service").map(i=>{const e=String(i.owner||"").trim(),n=h(e),o=b(i)||i.savedAt,r=d(i.contact,i.contactName,i.contactPhone);return{id:o,name:n,serviceTitle:String(i.title||i.serviceTitle||i.c

In [22]:
position = module_text.find("ne=")

print("Position:", position)
print(module_text[position:position + 2500])

Position: 5634
ne=async t=>{const i=(await S()).filter(e=>String(e.listingType||"equipment")==="equipment").map(e=>{const n=String(e.owner||"").trim(),o=String(e.title||"Equipment listing").trim(),r=b(e)||e.savedAt,s=d(e.contact,e.contactName,e.contactPhone),c=p(String(e.purchasePrice||e.price||"").trim()),l=p(String(e.pricePerDay||e.dailyRate||e.rate||"").trim(),"/day")||(c?"":"Contact owner"),m=p(String(e.monthlyRate||e.pricePerMonth||"").trim()),y=p(String(e.annualRate||e.pricePerYear||"").trim()),u=N(e);u&&(P(r,u),P(o,u));const A=String(e.leaseOptions||"").split(",").map(x=>x.trim()).filter(Boolean),C=A.length>0?A:c?["Lease to use","Lease to own"]:["Lease to use"],L=String(e.paymentOption||e.escrowPaymentOption||"").trim().toLowerCase(),O=String(e.escrowEnabled||"").toLowerCase()==="yes"&&L!=="direct-offline";return{id:r,title:o,image:u,galleryMedia:U(e),availability:String(e.availability||"Available now").trim()||"Available now",pricePerDay:l,monthlyRate:m,annualRate:y,purchasePri

In [23]:
import re

for match in re.finditer(r'\bS\s*=\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>', module_text):
    start = match.start()
    print(module_text[start:start + 1200])
    print("-" * 50)

S=async()=>{const t=Date.now();if(t-T<X&&g.length>0)return g;try{const i=await W({status:"approved",limit:500});return g=Array.isArray(i.listings)?i.listings:[],T=t,g}catch(i){return console.warn("Failed to fetch approved listings:",i),g}},b=t=>String(t.id||t.equipmentId||t.savedAt||"").trim(),N=t=>V(String(t.image||t.imageUrl||t.bannerUrl||"").trim()),U=t=>[...String(t.galleryMedia||t.gallery||"").split(","),...String(t.galleryVideos||"").split(",")].map(e=>e.trim()).map(e=>V(e)).filter(Boolean).filter((e,n,o)=>o.indexOf(e)===n),Q=t=>{const i=w(t).role;return i==="provider"?"provider":i==="oem"?"oem":"lessor"},_=()=>{const t=g.filter(i=>String(i.listingType||"equipment")==="equipment").map(i=>{const e=String(i.owner||"").trim(),n=String(i.title||"Equipment listing").trim(),o=b(i)||i.savedAt,r=d(i.contact,i.contactName,i.contactPhone),s=p(String(i.purchasePrice||i.price||"").trim()),c=p(String(i.pricePerDay||i.dailyRate||i.rate||"").trim(),"/day")||(s?"":"Contact owner"),l=p(String(i.m

In [24]:
print(module_text[:1800])

import{ar as P,i as w,as as $,c as I,P as R,at as W,g as V}from"./index-NxSyl00x.js";import{p as D}from"./equipmentSpecifications-ff83feaI.js";import{g as G,m as j,c as J}from"./listingCategories-mOVKeq5L.js";const v=t=>["true","1","yes"].includes(String(t||"").trim().toLowerCase()),k=5*60*60*1e3,E=35,K=50,M="elra:equipment:view-boost-v1",Y=()=>Math.floor(Math.random()*(K-E+1))+E,p=(t,i="")=>{if(!t)return t;const e=t.match(/\/(day|week|month|6 months|year)/i),n=e?e[0].toLowerCase():i,o=t.replace(/[₦N,\s]/gi,"").replace(/\/(day|week|month|6 months|year).*/i,""),r=Number(o);return!Number.isFinite(r)||r===0?t:`₦${Math.round(r).toLocaleString()}${n}`},a=(t,i=0)=>{const e=Number(String(t||"").replace(/[^0-9.-]/g,""));return Number.isFinite(e)?e:i},z=()=>{if(typeof window>"u")return{lastAppliedAt:Date.now(),boostsByListing:{}};try{const t=window.localStorage.getItem(M);if(!t)return{lastAppliedAt:Date.now(),boostsByListing:{}};const i=JSON.parse(t),e=i.boostsByListing&&typeof i.boostsByListin

In [25]:
position = module_text.find("W=")
print("Position:", position)
print(module_text[position:position + 1800])

Position: -1



In [26]:
print(module_text[:1800])

import{ar as P,i as w,as as $,c as I,P as R,at as W,g as V}from"./index-NxSyl00x.js";import{p as D}from"./equipmentSpecifications-ff83feaI.js";import{g as G,m as j,c as J}from"./listingCategories-mOVKeq5L.js";const v=t=>["true","1","yes"].includes(String(t||"").trim().toLowerCase()),k=5*60*60*1e3,E=35,K=50,M="elra:equipment:view-boost-v1",Y=()=>Math.floor(Math.random()*(K-E+1))+E,p=(t,i="")=>{if(!t)return t;const e=t.match(/\/(day|week|month|6 months|year)/i),n=e?e[0].toLowerCase():i,o=t.replace(/[₦N,\s]/gi,"").replace(/\/(day|week|month|6 months|year).*/i,""),r=Number(o);return!Number.isFinite(r)||r===0?t:`₦${Math.round(r).toLocaleString()}${n}`},a=(t,i=0)=>{const e=Number(String(t||"").replace(/[^0-9.-]/g,""));return Number.isFinite(e)?e:i},z=()=>{if(typeof window>"u")return{lastAppliedAt:Date.now(),boostsByListing:{}};try{const t=window.localStorage.getItem(M);if(!t)return{lastAppliedAt:Date.now(),boostsByListing:{}};const i=JSON.parse(t),e=i.boostsByListing&&typeof i.boostsByListin

In [27]:
for term in ['"approved"', "status:", "limit:", "at as"]:
    print(term, js_response.text.find(term))

"approved" 354460
status: 13630
limit: 34117
at as -1


In [28]:
position = js_response.text.find('"approved"')
print(js_response.text[position - 800:position + 1200])

p(n=>({id:n.id,documentCode:n.document_code,documentType:n.document_type,documentTypeLabel:n.document_type_label,mimeType:n.mime_type,verificationStatus:n.verification_status,verificationNotes:n.verification_notes,isRequired:n.is_required,fileUrl:n.file_url,createdAt:n.created_at,updatedAt:n.updated_at}))}},dF=async e=>{var n;const t=e.trim().toUpperCase();if(!t)return{verified:!1,status:"Not Verified",message:"Please enter a valid equipment registry ID.",checkedAt:new Date().toISOString(),registryId:"",source:"offline"};try{const r=await fetch(`${K2}/${encodeURIComponent(t)}`);if(!r.ok)throw new Error("Equipment registry verification API is not reachable.");const s=await r.json(),i=H2(s==null?void 0:s.data,t),o=String(((n=s==null?void 0:s.data)==null?void 0:n.status)||"").toLowerCase()==="approved",a=!!(s!=null&&s.success&&(s!=null&&s.data)&&o);return{verified:a,status:a?"Verified":"Not Verified",message:(s==null?void 0:s.message)||(a?`Equipment registry ID ${t} has been verified succ

In [29]:
print(js_response.text[-3000:])

w.scrollTo({top:0,left:0,behavior:"auto"})},[e]),null},Rz=()=>(m.useEffect(()=>{DE()==="accepted"&&NE()},[]),f.jsxs("div",{className:"pb-24 md:pb-0",children:[f.jsx(kz,{}),f.jsx(zj,{}),f.jsxs(pR,{children:[k4(),W4(),Az(),H4()]}),f.jsx(Vj,{}),f.jsx(Dj,{}),f.jsx(P2,{})]})),_z=()=>f.jsx(Tk,{client:Cz,children:f.jsx(SR,{children:f.jsxs(f2,{children:[f.jsx(xL,{}),f.jsx(o_,{}),f.jsx(Rz,{})]})})});typeof window<"u"&&window.addEventListener("vite:preloadError",e=>{e.preventDefault();const t="app_chunk_reload_timestamp",n=sessionStorage.getItem(t),r=Date.now();(!n||r-Number(n)>1e4)&&(sessionStorage.setItem(t,String(r)),window.location.reload())});Jb(document.getElementById("root")).render(f.jsx(_z,{}));export{GF as $,vF as A,_a as B,of as C,Rj as D,DU as E,cE as F,yo as G,LU as H,$n as I,mF as J,bF as K,xr as L,Xs as M,SF as N,EF as O,nF as P,RE as Q,_j as R,Bl as S,RU as T,og as U,_U as V,Ie as W,xm as X,hI as Y,jz as Z,S2 as _,xn as a,il as a$,YF as a0,JF as a1,j as a2,QF as a3,F3 as a4,ut as

In [30]:
position = js_response.text.find("D2=")
print("Position:", position)
print(js_response.text[position:position + 2500])

Position: 346617
D2=async(e={})=>{const t=yn(Mn);let n=null;for(const r of t)try{const s=await fetch(Ae(`/api/listings${L2(e)}`,r),{method:"GET",cache:"no-store",headers:{"Cache-Control":"no-cache"}}),i=await Ka(s);return Mn=r,i}catch(s){n=s instanceof Error?s:new Error("Listing request failed.")}throw n||new Error("Listing request failed.")},iF=async({id:e,owner:t,type:n,limit:r=200})=>{const s=String(e||"").trim();if(!s)return null;const i=await D2({owner:t,type:n,limit:r});return(Array.isArray(i.listings)?i.listings:[]).find(a=>String(a.id||"").trim()===s)||null},oF=async(e,t,n)=>{const r=yn(Mn);let s=null;for(const i of r)try{const o=await fetch(Ae(`/api/listings/${encodeURIComponent(t)}/moderation`,i),{method:"PATCH",headers:{"Content-Type":"application/json",Authorization:`Bearer ${e}`,"Cache-Control":"no-cache"},cache:"no-store",body:JSON.stringify(n)}),a=await Ka(o);return Mn=i,a}catch(o){s=o instanceof Error?o:new Error("Listing moderation request failed.")}throw s||new Error(

### 4. Testing the API:
Request a small page of approved listings, check the response structure and confirm pagination works.

In [31]:
import requests

api_url = "https://equipmentmarket.ng/api/listings"

response = requests.get(
    api_url,
    params={"status": "approved", "limit": 10},
    timeout=20
)

print("Status code:", response.status_code)
print("URL requested:", response.url)

Status code: 200
URL requested: https://equipmentmarket.ng/api/listings?status=approved&limit=10


In [32]:
data = response.json()

print("Success:", data.get("success"))
print("Response keys:", list(data.keys()))
print("Listings returned:", len(data.get("listings", [])))

if data.get("listings"):
    print("Available fields:")
    print(list(data["listings"][0].keys()))

print("Other response information:")
print({key: value for key, value in data.items()
       if key != "listings"})

Success: True
Response keys: ['success', 'listings', 'meta']
Listings returned: 10
Available fields:
['id', 'status', 'title', 'category', 'subcategory', 'categoryPath', 'dailyRate', 'monthlyRate', 'annualRate', 'salePrice', 'location', 'minimumLease', 'description', 'contact', 'contactName', 'contactPhone', 'condition', 'availability', 'equipmentSpecifications', 'equipmentSpecificationsJson', 'elraRegistrationId', 'image', 'mainMediaType', 'gallery', 'galleryMedia', 'galleryVideos', 'escrowEnabled', 'escrowPaymentOption', 'paymentOption', 'leaseOptions', 'downpaymentPercentage', 'isBnpl', 'provideOperator', 'operatorPolicy', 'isPreorder', 'estimatedDeliveryDate', 'deliveredFrom', 'isInsured', 'insuranceId', 'insuranceDocument', 'requireMembership', 'allowedAssociations', 'listingType', 'owner', 'savedAt', 'submissionType', 'adminDecisionNote', 'isFeatured', 'featuredDurationDays', 'views', 'interactionStats', 'equipmentId', 'verificationStatus', 'price', 'createdAt', 'updatedAt']
Othe

In [33]:
page2_response = requests.get(
    api_url,
    params={"status": "approved", "limit": 10, "page": 2},
    timeout=20
)

print("Status code:", page2_response.status_code)

page2_data = page2_response.json()
page2_listings = page2_data.get("listings", [])

print("Page 2 listings:", len(page2_listings))
print("Page 2 metadata:", page2_data.get("meta"))

print("\nFirst listing on page 1:", data["listings"][0]["title"])
print("First listing on page 2:", page2_listings[0]["title"])

Status code: 200
Page 2 listings: 10
Page 2 metadata: {'total': 245, 'page': 2, 'limit': 10, 'pageCount': 25, 'hasNextPage': True, 'hasPreviousPage': True}

First listing on page 1: Wuling Bingo Plus
First listing on page 2: 315 SANY Excavator


### 5. Collecting All Pages:
Loop through all 25 pages (10 listings each), pausing between requests.

In [34]:

import requests
import pandas as pd
import time

api_url = "https://equipmentmarket.ng/api/listings"

all_listings = []

print("Scraper is ready.")


Scraper is ready.


In [35]:

first_response = requests.get(
    api_url,
    params={"status": "approved", "limit": 10, "page": 1},
    timeout=20
)

first_response.raise_for_status()
first_data = first_response.json()

total_pages = first_data["meta"]["pageCount"]

print("Total listings:", first_data["meta"]["total"])
print("Total pages:", total_pages)


Total listings: 245
Total pages: 25


In [36]:

all_listings.extend(first_data["listings"])

for page in range(2, total_pages + 1):
    page_response = requests.get(
        api_url,
        params={
            "status": "approved",
            "limit": 10,
            "page": page
        },
        timeout=20
    )

    page_response.raise_for_status()
    page_data = page_response.json()

    all_listings.extend(page_data.get("listings", []))

    print(
        f"Page {page}/{total_pages}: "
        f"{len(page_data.get('listings', []))} listings collected"
    )

    time.sleep(1)

print("Total records collected:", len(all_listings))


Page 2/25: 10 listings collected
Page 3/25: 10 listings collected
Page 4/25: 10 listings collected
Page 5/25: 10 listings collected
Page 6/25: 10 listings collected
Page 7/25: 10 listings collected
Page 8/25: 10 listings collected
Page 9/25: 10 listings collected
Page 10/25: 10 listings collected
Page 11/25: 10 listings collected
Page 12/25: 10 listings collected
Page 13/25: 10 listings collected
Page 14/25: 10 listings collected
Page 15/25: 10 listings collected
Page 16/25: 10 listings collected
Page 17/25: 10 listings collected
Page 18/25: 10 listings collected
Page 19/25: 10 listings collected
Page 20/25: 10 listings collected
Page 21/25: 10 listings collected
Page 22/25: 10 listings collected
Page 23/25: 10 listings collected
Page 24/25: 10 listings collected
Page 25/25: 5 listings collected
Total records collected: 245


### 6. Saving the Dataset:
Save only the fields needed for the analysis. Seller contact and owner fields are left out for privacy.

In [37]:
import json

fields_to_keep = [
    "id", "title", "category", "subcategory", "location", "condition",
    "availability", "minimumLease", "dailyRate", "monthlyRate", "annualRate",
    "salePrice", "createdAt", "updatedAt", "listingType",
    "verificationStatus", "isFeatured", "views"
]

public_listings = [
    {field: listing.get(field) for field in fields_to_keep}
    for listing in all_listings
]

with open("data/equipment_listings_raw.json", "w", encoding="utf-8") as f:
    json.dump(public_listings, f, ensure_ascii=False)

print("Saved", len(public_listings), "listings with", len(fields_to_keep), "fields")

Saved 245 listings with 18 fields
